# Global Factor Premiums

**Authors:** Guido Baltussen, Laurens Swinkels, Pim van Vliet

**Published:** 2019-01-01

**URL:** [https://doi.org/10.2139/ssrn.3325720](https://doi.org/10.2139/ssrn.3325720)

**Abstract:** This notebook is inspired by the paper 'Global Factor Premiums' and aims to explore factor-based investment strategies using historical market data. We will follow a structured approach using the CRISP-TIQ 6-phase methodology.

In [ ]:
!pip install yfinance

## Phase 1: Configuration

In this phase, we set up the initial configuration for our analysis, including the list of tickers, the time period for analysis, and any other parameters required.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Configuration
TICKERS = ['AAPL', 'MSFT', 'GOOGL']
START_DATE = '2015-01-01'
END_DATE = '2023-01-01'
RISK_FREE_RATE = 0.01

## Phase 2: Data Download and Feature Engineering

In this phase, we download historical price data and engineer features that will be used for signal generation.

In [ ]:
# Download data
prices = yf.download(TICKERS, start=START_DATE, end=END_DATE)['Adj Close']

# Calculate returns
returns = prices.pct_change().dropna()

## Phase 3: Signal Generation and Portfolio Construction

Generate trading signals based on the features and construct a portfolio.

In [ ]:
# Simple moving average crossover strategy
short_window = 40
long_window = 100

signals = pd.DataFrame(index=prices.index)
signals['signal'] = 0.0
signals['short_mavg'] = prices.rolling(window=short_window, min_periods=1, center=False).mean()
signals['long_mavg'] = prices.rolling(window=long_window, min_periods=1, center=False).mean()

# Generate signals
signals['signal'][short_window:] = np.where(signals['short_mavg'][short_window:] > signals['long_mavg'][short_window:], 1.0, 0.0)

# Generate trading orders
signals['positions'] = signals['signal'].diff()

## Phase 4: Vectorized Backtest

Conduct a backtest of the strategy using vectorized operations to shift signals and avoid look-ahead bias.

In [ ]:
# Backtest
portfolio = pd.DataFrame(index=signals.index)
portfolio['positions'] = signals['signal'].shift(1) * returns
portfolio['total'] = portfolio['positions'].cumsum()

## Phase 5: Performance Metrics

Calculate performance metrics such as Sharpe ratio, Sortino ratio, Calmar ratio, maximum drawdown, and plot the equity curve.

In [ ]:
# Performance metrics
def calculate_sharpe_ratio(returns, risk_free_rate):
    excess_returns = returns - risk_free_rate
    return np.sqrt(252) * excess_returns.mean() / excess_returns.std()

def calculate_sortino_ratio(returns, risk_free_rate):
    excess_returns = returns - risk_free_rate
    downside_std = np.std(excess_returns[excess_returns < 0])
    return np.sqrt(252) * excess_returns.mean() / downside_std

def calculate_max_drawdown(portfolio_value):
    roll_max = portfolio_value.cummax()
    drawdown = portfolio_value - roll_max
    return drawdown.min()

sharpe_ratio = calculate_sharpe_ratio(portfolio['positions'], RISK_FREE_RATE)
sortino_ratio = calculate_sortino_ratio(portfolio['positions'], RISK_FREE_RATE)
max_drawdown = calculate_max_drawdown(portfolio['total'])

# Plot equity curve
plt.figure(figsize=(10, 6))
plt.plot(portfolio['total'], label='Portfolio Value')
plt.title('Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.legend()
plt.show()

print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Sortino Ratio: {sortino_ratio:.2f}")
print(f"Max Drawdown: {max_drawdown:.2f}")

## Phase 6: Monitoring Stub

This phase is reserved for implementing monitoring tools for the strategy, such as alerts for significant drawdowns or deviations from expected performance.

In [ ]:
# Monitoring stub
# Placeholder for future monitoring implementation
print("Monitoring tools to be implemented.")